In [1]:
import anthropic
import config
import logging

logger = logging.getLogger(__name__)

_client = None

In [2]:
def get_client() -> anthropic.Anthropic:
    global _client
    if _client is None:
        _client = anthropic.Anthropic(api_key=config.ANTHROPIC_API_KEY)
    return _client

In [3]:
HYDE_SYSTEM = """You are an expert in Toronto municipal by-laws and Ontario tenancy law.
Given a question from a Toronto resident, write a short passage (3-5 sentences) that
looks like it could appear in an official Toronto by-law or Ontario legislation document
and would directly answer the question.

Write only the passage itself — no preamble, no explanation, no section numbers.
Use formal legal language consistent with municipal by-laws."""


def hyde_expand(query):
    client = get_client()

    response = client.messages.create(
        model=config.HYDE_MODEL,
        max_tokens=200,
        system=HYDE_SYSTEM,
        messages=[{"role": "user", "content": query}],
    )

    expanded = response.content[0].text.strip()
    logger.info(f"HyDE expansion: {expanded[:80]}...")
    return expanded

In [9]:
expanded_query = hyde_expand("What are the quiet hours for construction noise in Toronto?")
print(expanded_query)

Construction activities that produce noise shall not be carried out between the hours of 7:00 p.m. and 7:00 a.m. on weekdays, nor between the hours of 7:00 p.m. on Friday and 9:00 a.m. on Saturday, nor between the hours of 7:00 p.m. on Saturday and 9:00 a.m. on Sunday, except as otherwise permitted by an exemption or variance issued by the City. For the purposes of this provision, construction activities shall include, but are not limited to, the operation of power tools, heavy equipment, and machinery associated with building, renovation, or demolition work. The restrictions set out herein shall not apply to emergency repair work necessary to prevent injury, loss of life, or damage to property, provided that the owner or contractor notify the City at the earliest practicable opportunity.


In [12]:
from retrieval.retrieve import hybrid_search
result = hybrid_search(expanded_query, top_k=5, domain="noise")

In [13]:
result

[RetrievedChunk(chunk_id='noise::591-2.3::0', domain='noise', section_id='591-2.3', section_title='Construction.', parent_section='ARTICLE 2', text='[Amended 2024-03-22 by By-law 268-2024] No person shall emit or cause or permit the emission of sound resulting from construction or any operation of construction equipment that is clearly audible: A. From 7 p.m. to 7 a.m. the next day, except until 9 a.m. on Saturdays; and/or B. All day on Sundays and statutory holidays.', page=6, score=0.03252247488101534),
 RetrievedChunk(chunk_id='noise::591-2.4::0', domain='noise', section_id='591-2.4', section_title='Loading and unloading.', parent_section='ARTICLE 2', text='[Amended 2019-07-18 by By-law 1102-2019; 2022-07-22 by By-law 1102-2022] A. No person shall emit or cause or permit the emission of sound resulting from loading, unloading, delivering, packing, unpacking, and otherwise handling any containers, products or materials from 11 p.m. to 7 a.m. the next day, except until 9 a.m. on Satur